project title : Next word prediction
Steps:
1. data collection: Shakespher's "hamlet"
2. data processing: tokenized and converted and sequence and padded to to ensure uniform length
3. model buolding: An LSTM model is constructed with embedding layer, two lstm layers, and a dense output layer with softmax activation function to predict next word
4. model trainig: trainde using prepared sequence, with early stopping to ensure overfitting. Early stopping monitores the validation loss and stops trainnig when the loss stops improving
5. model evaluation: the model is evaluated using some examples sentences to test the ability to predict next word
6. deployment: A streamlite web application is  developed to allow to allow users to input a sequence of words and get the predicted next word in real time.

requirements :
tensorflow==2.15.0
pandas
numpy
scikit-learn
tensorboard
matplotlib
streamlit
scikeras
nltk

In [1]:
#installing requirements
!python --version

!pip install tensorflow
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install tensorboard
!pip install matplotlib
!pip install streamlit
!pip install scikeras
!pip install nltk

Python 3.13.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 121.2 MB/s eta 0:00:00


In [3]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
!nvidia-smi

TensorFlow version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Wed Oct  7 19:59:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                       

In [4]:
# data collection
import nltk
nltk.download("gutenberg")
from nltk.corpus import gutenberg
import pandas as pd

# load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')

# save toa file
with open('hamlet.txt','w') as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [7]:
# data preprocessing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import  train_test_split

# load the dataset
with open('hamlet.txt','r') as file:
    data = file.read().lower()

# tokenize the text - creating indexes
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index)+1
total_words



4818

In [8]:
# input sequences
input_sequences = []
for line in data.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

In [9]:
input_sequences

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [11]:
# pad sequence
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences,max_sequence_len,padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [14]:
## create predictors and label
X,y = input_sequences[:,:-1],input_sequences[:,-1]
y

array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32)

In [16]:
import tensorflow as tf

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [18]:
# train test split
X_train, X_test,y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [25]:
# train LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout,Input

# define model
model = Sequential()
model.add(Input(shape=(max_sequence_len,)))
model.add(Embedding(
    input_dim= total_words,
    output_dim = 100
))
model.add(LSTM(150,return_sequences = True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation = 'softmax'))

#compile the model
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()



Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 14, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 14, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
from re import VERBOSE
# train the model
history = model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test))

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.0390 - loss: 6.4737 - val_accuracy: 0.0406 - val_loss: 6.7573
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0472 - loss: 6.3359 - val_accuracy: 0.0470 - val_loss: 6.8046
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.0509 - loss: 6.1890 - val_accuracy: 0.0515 - val_loss: 6.7955
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.0563 - loss: 6.0460 - val_accuracy: 0.0571 - val_loss: 6.8281
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0615 - loss: 5.8954 - val_accuracy: 0.0579 - val_loss: 6.9107
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0677 - loss: 5.7591 - val_accuracy: 0.0626 - val_loss: 6.9520
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0753 - loss: 5.6147 - val_accuracy: 0.0670 - val_loss: 7.0231
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0825 - loss: 5.4737 - val_accu

In [30]:
# function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
  token_list= tokenizer.texts_to_sequences([text])[0]
  if len(token_list)>=max_sequence_len:
    token_list = token_list[-(max_sequence_len):]
  token_list = pad_sequences([token_list],maxlen=max_sequence_len,padding='pre')
  predicted = model.predict(token_list)
  predicted_word_index = np.argmax(predicted,axis=1)
  for word, index in tokenizer.word_index.items():
    if index == predicted_word_index:
      return word
  return None

In [33]:
input_text = "to be or not to be that is the"
print(f'input text:{input_text}')
max_sequence_len= model.input_shape[1]
next_word = predict_next_word(model, tokenizer,input_text, max_sequence_len)
print(f"Next word prediction: {next_word}")

input text:to be or not to be that is the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Next word prediction: king


In [36]:
# save the model
model.save("next_word_lstm.h5")
# save tokenizer
import pickle
with open("tokenizer.pickle",'wb') as file:
  pickle.dump(tokenizer,file, protocol=pickle.HIGHEST_PROTOCOL)

In [37]:
input_text = "Bar. Long liue the"
print(f'input text:{input_text}')
max_sequence_len= model.input_shape[1]
next_word = predict_next_word(model, tokenizer,input_text, max_sequence_len)
print(f"Next word prediction: {next_word}")

input text:Bar. Long liue the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Next word prediction: king


In [40]:
# steamlite web app
import streamlit as st
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# load the model
model = load_model('next_word_lstm.h5')

# load the pickel file
with open('tokenizer.pickle','rb') as file:
  tokenizer = pickle.load(file)

# function to predict next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
  token_list= tokenizer.texts_to_sequences([text])[0]
  if len(token_list)>=max_sequence_len:
    token_list = token_list[-(max_sequence_len):]
  token_list = pad_sequences([token_list],maxlen=max_sequence_len,padding='pre')
  predicted = model.predict(token_list)
  predicted_word_index = np.argmax(predicted,axis=1)
  for word, index in tokenizer.word_index.items():
    if index == predicted_word_index:
      return word
  return None


In [ ]:
## streamlite app
st.title("Next word Prediction With LSTM")
input_text=st.text_input("Enter the sequence of words",'to be or not to be')
if st.button("predict Next Word"):
  max_sequence_len= model.input_shape[1]
  next_word = predict_next_word(model, tokenizer,input_text, max_sequence_len)
  st.write(f'Next word:{next_word}')